# Lab 8: Tree-Based Methods


## Imports


In [17]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots
import sklearn.model_selection as skm
from ISLP import load_data, confusion_table
from ISLP.models import ModelSpec as MS


In [18]:
from sklearn.tree import (DecisionTreeClassifier as DTC,
                          DecisionTreeRegressor as DTR,
                          plot_tree,
                          export_text)
from sklearn.metrics import (accuracy_score,
                             log_loss)
from sklearn.ensemble import (RandomForestRegressor as RF,
                              GradientBoostingRegressor as GBR,
                              GradientBoostingClassifier as GBC)


## Question 1: Bagging on Hitters dataset
Using the Hitters dataset (remove rows with missing salary values), split the data into 70% training and 30% test sets with random_state=42. Fit a bagging model (Random Forest with max_features equal to the number of predictors) using 300 trees and random_state=42 to predict Salary. Report the test set MSE rounded to 2 decimal places.


In [19]:
# Load and prepare Hitters dataset
Hitters = load_data('Hitters')
Hitters = Hitters.dropna(subset=['Salary'])
print(f"Hitters data shape: {Hitters.shape}")


Hitters data shape: (263, 20)


In [20]:
# Create model matrix
y = Hitters['Salary'].values
model = MS(Hitters.columns.drop('Salary'), intercept=False)
D = model.fit_transform(Hitters)
feature_names = list(D.columns)
X = np.asarray(D)
print(f"Number of features: {X.shape[1]}")


Number of features: 19


In [21]:
# Split data
X_train, X_test, y_train, y_test = skm.train_test_split(
    X, y, test_size=0.3, random_state=42
)
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 184
Test set size: 79


In [22]:
# Fit bagging model (max_features = number of predictors)
n_features = X_train.shape[1]
bagging_model = RF(max_features=n_features, n_estimators=300, random_state=42)
bagging_model.fit(X_train, y_train)

# Predict and compute test MSE
y_pred = bagging_model.predict(X_test)
test_mse = np.mean((y_test - y_pred) ** 2)
print(f"Question 1 - Test MSE: {test_mse:.2f}")


Question 1 - Test MSE: 121187.76


## Question 2: Random Forest on Hitters dataset
Using the same train/test split from Question 1 on the Hitters dataset, fit a Random Forest model with max_features=5, 300 trees, and random_state=42. Create a DataFrame showing the feature importance values sorted in descending order by importance. Which variable has the highest importance score?


In [23]:
# Fit Random Forest with max_features=5
rf_model = RF(max_features=5, n_estimators=300, random_state=42)
rf_model.fit(X_train, y_train)

# Create DataFrame with feature importances
feature_imp_df = pd.DataFrame({
    'importance': rf_model.feature_importances_
}, index=feature_names)
feature_imp_df = feature_imp_df.sort_values(by='importance', ascending=False)
print("Question 2 - Feature Importances:")
print(feature_imp_df)
print(f"\nHighest importance variable: {feature_imp_df.index[0]}")
print(f"Importance score: {feature_imp_df.iloc[0, 0]:.4f}")


Question 2 - Feature Importances:
              importance
CHits           0.141980
CAtBat          0.117098
CRBI            0.113280
CRuns           0.101749
CHmRun          0.074995
CWalks          0.067667
PutOuts         0.057813
RBI             0.051790
AtBat           0.050193
Runs            0.047145
Hits            0.045581
Walks           0.037173
Years           0.032647
HmRun           0.025920
Assists         0.014068
Errors          0.011789
NewLeague[N]    0.003421
Division[W]     0.002981
League[N]       0.002711

Highest importance variable: CHits
Importance score: 0.1420


## Question 3: Decision Tree on OJ dataset
Using the OJ dataset, fit a decision tree classifier with criterion='entropy', max_depth=4, min_samples_leaf=5, and random_state=42 using all variables except Purchase to predict Purchase. Split the data 75% training and 25% test with random_state=42. What is the test accuracy (proportion of correct predictions) rounded to 3 decimal places?


In [24]:
# Load OJ dataset
OJ = load_data('OJ')
print(f"OJ data shape: {OJ.shape}")


OJ data shape: (1070, 18)


In [25]:
# Create model matrix
y_oj = OJ['Purchase'].values
model_oj = MS(OJ.columns.drop('Purchase'), intercept=False)
D_oj = model_oj.fit_transform(OJ)
feature_names_oj = list(D_oj.columns)
X_oj = np.asarray(D_oj)
print(f"Number of features: {X_oj.shape[1]}")


Number of features: 17


In [26]:
# Split data 75/25
X_train_oj, X_test_oj, y_train_oj, y_test_oj = skm.train_test_split(
    X_oj, y_oj, test_size=0.25, random_state=42
)
print(f"Training set size: {X_train_oj.shape[0]}")
print(f"Test set size: {X_test_oj.shape[0]}")


Training set size: 802
Test set size: 268


In [27]:
# Fit decision tree
dtc_oj = DTC(criterion='entropy', max_depth=4, min_samples_leaf=5, random_state=42)
dtc_oj.fit(X_train_oj, y_train_oj)

# Compute test accuracy
y_pred_oj = dtc_oj.predict(X_test_oj)
test_accuracy = accuracy_score(y_test_oj, y_pred_oj)
print(f"Question 3 - Test accuracy: {test_accuracy:.3f}")


Question 3 - Test accuracy: 0.817


## Question 4: Gradient Boosting on Auto dataset
Using the Auto dataset (remove rows with missing values), create a binary variable mpg_high that equals 1 when mpg is above the median and 0 otherwise. Split the data 70/30 train/test (random_state=42). Fit a Gradient Boosting Classifier with n_estimators=200, learning_rate=0.1, max_depth=3, and random_state=42 using all variables except mpg and name. Report the test set accuracy rounded to 3 decimal places.


In [28]:
# Load Auto dataset
Auto = load_data('Auto')
Auto = Auto.dropna()
print(f"Auto data shape: {Auto.shape}")
print(f"\nMedian mpg: {Auto['mpg'].median():.2f}")


Auto data shape: (392, 8)

Median mpg: 22.75


In [29]:
# Create binary variable mpg_high
Auto['mpg_high'] = np.where(Auto['mpg'] > Auto['mpg'].median(), 1, 0)
print(f"mpg_high distribution:\n{Auto['mpg_high'].value_counts()}")


mpg_high distribution:
mpg_high
0    196
1    196
Name: count, dtype: int64


In [30]:
# Create model matrix (excluding mpg and mpg_high)
model_auto = MS(Auto.columns.drop(['mpg', 'mpg_high']), intercept=False)
D_auto = model_auto.fit_transform(Auto)
feature_names_auto = list(D_auto.columns)
X_auto = np.asarray(D_auto)
y_auto = Auto['mpg_high'].values
print(f"Number of features: {X_auto.shape[1]}")


Number of features: 7


In [31]:
# Split data 70/30
X_train_auto, X_test_auto, y_train_auto, y_test_auto = skm.train_test_split(
    X_auto, y_auto, test_size=0.3, random_state=42
)
print(f"Training set size: {X_train_auto.shape[0]}")
print(f"Test set size: {X_test_auto.shape[0]}")


Training set size: 274
Test set size: 118


In [32]:
# Fit Gradient Boosting Classifier
gbc_model = GBC(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42)
gbc_model.fit(X_train_auto, y_train_auto)

# Compute test accuracy
y_pred_auto = gbc_model.predict(X_test_auto)
test_accuracy_auto = accuracy_score(y_test_auto, y_pred_auto)
print(f"Question 4 - Test accuracy: {test_accuracy_auto:.3f}")


Question 4 - Test accuracy: 0.890


## Question 5: Pruned Regression Tree on Hitters dataset
Using the Hitters dataset (remove missing values) with a 70/30 train/test split (random_state=42), fit a regression tree with max_depth=6 and random_state=42 to predict Salary. Then use cost_complexity_pruning_path() and GridSearchCV with 5-fold cross-validation (random_state=42) to find the optimal ccp_alpha value. Report both: (a) the number of leaf nodes in the best estimator from the grid search, and (b) the test MSE of the pruned tree rounded to 2 decimal places.


In [33]:
# Fit initial regression tree with max_depth=6
reg_tree = DTR(max_depth=6, random_state=42)
reg_tree.fit(X_train, y_train)

# Get cost complexity pruning path
ccp_path = reg_tree.cost_complexity_pruning_path(X_train, y_train)
print(f"Number of ccp_alpha values: {len(ccp_path.ccp_alphas)}")


Number of ccp_alpha values: 40


In [34]:
# 5-fold cross-validation
kfold = skm.KFold(5, shuffle=True, random_state=42)
grid = skm.GridSearchCV(
    reg_tree,
    {'ccp_alpha': ccp_path.ccp_alphas},
    refit=True,
    cv=kfold,
    scoring='neg_mean_squared_error'
)
grid.fit(X_train, y_train)

# Get best estimator
best_tree = grid.best_estimator_
num_leaves = best_tree.tree_.n_leaves

# Predict on test set and compute MSE
y_pred_q5 = best_tree.predict(X_test)
test_mse_q5 = np.mean((y_test - y_pred_q5) ** 2)

print(f"Question 5:")
print(f"(a) Number of leaf nodes: {num_leaves}")
print(f"(b) Test MSE: {test_mse_q5:.2f}")


Question 5:
(a) Number of leaf nodes: 4
(b) Test MSE: 161392.55
